# Week 21 · Notebook 2: Medallion in SQL (bronze → silver → gold)

# Requirements: Databricks workspace (free trial) + upload the week-01 CSVs to a volume

Upload into `/Volumes/zrl_/zorologistics/raw/`:

- `shipments.csv`
- `carriers.csv`
- `lanes.csv`

Run this on a SQL warehouse (or an all-purpose cluster) after `01-unity-catalog-and-delta-lab.ipynb` has created the catalog/schema/volume.


## The medallion architecture, in SQL

**Bronze** preserves raw source fidelity (append-only). **Silver** cleans: dedupe on `shipment_id`, cast timestamps, impute the `NULL` weights/distances planted in Week 2. **Gold** aggregates into business shapes (`gold_on_time_kpis` by carrier/lane/month). The discipline from Week 2 becomes declarative SQL here, and the tables are Delta, so every write is versioned and time-travelable.


In [ ]:
%sql
-- Work in our catalog/schema so names stay short below.
USE CATALOG zrl_;
USE SCHEMA zorologistics;


In [ ]:
%sql
-- Bronze: raw, unvalidated, source fidelity. Load the three CSVs as Delta tables.
CREATE OR REPLACE TABLE shipments_bronze AS
SELECT *
FROM read_files('/Volumes/zrl_/zorologistics/raw/shipments.csv',
                format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE carriers AS
SELECT *
FROM read_files('/Volumes/zrl_/zorologistics/raw/carriers.csv',
                format => 'csv', header => true, inferSchema => true);

CREATE OR REPLACE TABLE lanes AS
SELECT *
FROM read_files('/Volumes/zrl_/zorologistics/raw/lanes.csv',
                format => 'csv', header => true, inferSchema => true);


## Silver: dedupe, cast, impute

The Week-2 generator plants duplicates (0.2%) and `NaN` weights (0.3%); the lanes file plants `NaN` distances (5%). In CSV those `NaN`s became empty strings, which `read_files` turns into `NULL`. We (1) dedupe with `ROW_NUMBER()` over `shipment_id`, (2) cast timestamps with an optional-fraction format, and (3) impute `weight_kg` and `distance_km` with their means.


In [ ]:
%sql
-- Silver: dedupe on shipment_id, cast timestamps, impute nulls, recompute is_on_time.
CREATE OR REPLACE TABLE silver_shipments AS
WITH dedup AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY shipment_id ORDER BY planned_departure) AS rn
  FROM shipments_bronze
)
SELECT
  shipment_id,
  carrier_id,
  lane_id,
  commodity,
  COALESCE(CAST(weight_kg AS DOUBLE), 850.0) AS weight_kg,
  COALESCE(CAST(value_usd AS DOUBLE), 0.0)  AS value_usd,
  to_timestamp(planned_departure, 'yyyy-MM-dd HH:mm:ss[.SSSSSS]') AS planned_departure,
  to_timestamp(planned_arrival,   'yyyy-MM-dd HH:mm:ss[.SSSSSS]') AS planned_arrival,
  to_timestamp(actual_arrival,    'yyyy-MM-dd HH:mm:ss[.SSSSSS]') AS actual_arrival,
  COALESCE(CAST(delay_hours AS DOUBLE), 0.0) AS delay_hours,
  CASE WHEN CAST(delay_hours AS DOUBLE) <= 2.0 THEN true ELSE false END AS is_on_time,
  status,
  weather_severity
FROM dedup
WHERE rn = 1


In [ ]:
%sql
-- Silver quality check: no nulls in the imputed columns, no duplicate shipment_ids.
SELECT
  count(*)                        AS rows,
  count(*) FILTER (WHERE weight_kg IS NULL)  AS null_weights,
  count(DISTINCT shipment_id)     AS distinct_shipments
FROM silver_shipments


In [ ]:
%sql
-- Impute the lanes distance_km the same way (Week 2 planted NaN distances here).
CREATE OR REPLACE TABLE lanes_clean AS
SELECT
  lane_id, origin, destination,
  COALESCE(CAST(distance_km AS DOUBLE), 900.0) AS distance_km,
  avg_transit_days, toll_km, port_region
FROM lanes


## Gold: on-time KPIs by carrier / lane / month

Gold is dimensional and business-aligned, what BI and analysts actually query. We join silver to the cleaned carrier/lane dimensions and aggregate at the carrier×lane×month grain.


In [ ]:
%sql
-- Gold: business-aligned aggregates for BI and ML.
CREATE OR REPLACE TABLE gold_on_time_kpis AS
SELECT
  s.carrier_id,
  c.carrier_name,
  s.lane_id,
  l.origin,
  l.destination,
  date_format(s.actual_arrival, 'yyyy-MM') AS month,
  count(*)                                  AS shipment_count,
  round(avg(CASE WHEN s.is_on_time THEN 1.0 ELSE 0.0 END), 4) AS on_time_rate,
  round(avg(s.delay_hours), 2)              AS avg_delay_hours,
  round(sum(s.value_usd), 2)                AS total_value_usd,
  round(sum(s.weight_kg), 0)                AS total_weight_kg
FROM silver_shipments s
JOIN carriers c  ON s.carrier_id = c.carrier_id
JOIN lanes_clean l ON s.lane_id = l.lane_id
GROUP BY 1, 2, 3, 4, 5, 6


In [ ]:
%sql
-- Where do we lose money? Worst lanes/carriers by on-time rate first.
SELECT lane_id, origin, destination, month, shipment_count, on_time_rate, avg_delay_hours
FROM gold_on_time_kpis
ORDER BY on_time_rate ASC
LIMIT 10


## An AI/BI dashboard query

AI/BI dashboards (formerly Lakeview dashboards) consume exactly this kind of dataset. The query below is dashboard-ready: a monthly on-time trend with a 3-month rolling average. In the UI, run it in the SQL editor and click **"Create dashboard"** to publish it.


In [ ]:
%sql
-- AI/BI dashboard query: on-time rate by month with a 3-month rolling average.
SELECT
  month,
  on_time_rate,
  avg(on_time_rate) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS rolling_3m
FROM (
  SELECT month, round(avg(on_time_rate), 4) AS on_time_rate
  FROM gold_on_time_kpis
  GROUP BY month
)
ORDER BY month


In [ ]:
# Final metric: overall on-time rate and the gold grain count.
overall = spark.sql(
    "SELECT round(avg(CASE WHEN is_on_time THEN 1.0 ELSE 0.0 END), 4) "
    "FROM zrl_.zorologistics.silver_shipments").collect()[0][0]
gold_rows = spark.sql("SELECT count(*) FROM zrl_.zorologistics.gold_on_time_kpis").collect()[0][0]
print("overall on-time rate:", overall)
print("gold rows:", gold_rows)
